# Generate DWS-Bench thesis figures

This notebook generates the six core thesis figures from the stored model prediction artifacts:

1. Overall model x prompting accuracy
2. RQ1 accuracy by temporal depth
3. RQ2 revision accuracy heatmap
4. RQ3 distractor accuracy heatmap
5. Final-answer versus step-wise accuracy
6. RQ5 operation-family accuracy heatmap

The optional factor-correlation figure is disabled by default because it should only be used after the
factor statistics have been reviewed for the thesis.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent
if REPO_ROOT.name != 'StateMachine':
    REPO_ROOT = Path('/home/rahatut/Desktop/Projects/Thesis/StateMachine')
RESULTS_ROOT = REPO_ROOT / 'results'
OUTPUT_DIR = REPO_ROOT / 'report' / 'images' / 'thesis_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAKE_FACTOR_CORRELATION = False
DPI = 300

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 120,
})

CONFIGS = [
    ('0.5B', 'No CoT', 'dws_qwen05_no_cot'),
    ('0.5B', 'CoT', 'dws_qwen05_cot'),
    ('3B', 'No CoT', 'dws_qwen3b_no_cot'),
    ('3B', 'CoT', 'dws_qwen3b_cot'),
    ('7B', 'No CoT', 'dws_qwen7b_no_cot'),
    ('7B', 'CoT', 'dws_qwen7b_cot'),
]
CONFIG_LABELS = [f'Qwen2.5-{size} {prompt}' for size, prompt, _ in CONFIGS]
COLORS = {'No CoT': '#0F766E', 'CoT': '#E07A5F'}

In [ ]:
def choose_prediction_file(config_dir):
    candidates = sorted((RESULTS_ROOT / config_dir).glob('**/full_benchmark_predictions.jsonl'))
    if not candidates:
        raise FileNotFoundError(f'No prediction artifact found for {config_dir}')
    preferred = [path for path in candidates if '_256' in str(path)]
    return preferred[0] if preferred else candidates[0]


def load_prediction_file(path, model_size, prompting):
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            row['model_size'] = model_size
            row['prompting'] = prompting
            row['config_label'] = f'Qwen2.5-{model_size} {prompting}'
            row['requested_factors'] = row.get('requested_factors') or {}
            row['measured_factors'] = row.get('measured_factors') or {}
            rows.append(row)
    return rows


all_rows = []
selected_files = {}
for model_size, prompting, config_dir in CONFIGS:
    path = choose_prediction_file(config_dir)
    selected_files[(model_size, prompting)] = path
    all_rows.extend(load_prediction_file(path, model_size, prompting))

predictions = pd.DataFrame(all_rows)
predictions['is_correct'] = predictions['is_correct'].astype(bool)
predictions['step_accuracy'] = pd.to_numeric(predictions.get('step_accuracy'), errors='coerce')
predictions['experiment'] = predictions['experiment'].fillna('')
predictions['family'] = predictions['family'].fillna('')


def factor(frame, name):
    requested = frame['requested_factors'].map(lambda value: value.get(name, np.nan))
    measured = frame['measured_factors'].map(lambda value: value.get(f'{name}_actual', np.nan))
    return pd.to_numeric(requested).fillna(pd.to_numeric(measured))


for name in ['E', 'T', 'D', 'V', 'L_word']:
    predictions[name] = factor(predictions, name)

print('Loaded rows:', len(predictions))
for key, path in selected_files.items():
    print(f'{key[0]} {key[1]} -> {path.relative_to(REPO_ROOT)}')

In [ ]:
def save_figure(fig, filename):
    png_path = OUTPUT_DIR / f'{filename}.png'
    pdf_path = OUTPUT_DIR / f'{filename}.pdf'
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Wrote {png_path.relative_to(REPO_ROOT)} and {pdf_path.relative_to(REPO_ROOT)}')


def add_bar_labels(ax, bars):
    for bar in bars:
        value = bar.get_height()
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.015,
                f'{value:.1%}',
                ha='center',
                va='bottom',
                fontsize=8,
            )


def accuracy_by(frame, group_columns):
    grouped = frame.groupby(group_columns, dropna=False)['is_correct'].mean()
    return grouped.reset_index(name='accuracy')

In [ ]:
rq2 = predictions.query("experiment == 'rq2_revision'").copy()
rq2_summary = accuracy_by(rq2, ['config_label', 'T'])
rq2_table = rq2_summary.pivot(index='config_label', columns='T', values='accuracy').reindex(CONFIG_LABELS)
fig, ax = plt.subplots(figsize=(8.6, 4.8))
image = ax.imshow(rq2_table * 100, cmap='YlGnBu', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(rq2_table.columns)), [f'T={int(value)}' for value in rq2_table.columns])
ax.set_yticks(range(len(rq2_table.index)), rq2_table.index)
for row_index in range(rq2_table.shape[0]):
    for column_index in range(rq2_table.shape[1]):
        value = rq2_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('Revision-sweep target depth, $T$')
ax.set_title('RQ2: Revision accuracy heatmap', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig03_rq2_revision_heatmap')

In [ ]:
overall = accuracy_by(predictions, ['model_size', 'prompting'])
sizes = ['0.5B', '3B', '7B']
x = np.arange(len(sizes))
width = 0.34
fig, ax = plt.subplots(figsize=(7.4, 4.8))
for offset, prompting in [(-width / 2, 'No CoT'), (width / 2, 'CoT')]:
    values = [
        overall.query('model_size == @size and prompting == @prompting')['accuracy'].iloc[0]
        for size in sizes
    ]
    bars = ax.bar(x + offset, values, width, label=prompting, color=COLORS[prompting])
    add_bar_labels(ax, bars)
ax.set_xticks(x, sizes)
ax.set_ylim(0, 0.58)
ax.set_xlabel('Qwen2.5 parameter scale')
ax.set_ylabel('Final-answer accuracy')
ax.set_title('Overall accuracy by model scale and prompting', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2)
fig.tight_layout()
save_figure(fig, 'fig01_overall_model_prompting')

In [ ]:
rq1 = predictions.query("experiment == 'rq1_depth'").copy()
rq1_summary = accuracy_by(rq1, ['config_label', 'model_size', 'prompting', 'T'])
fig, ax = plt.subplots(figsize=(9.2, 5.4))
for label in CONFIG_LABELS:
    subset = rq1_summary.query('config_label == @label').sort_values('T')
    if subset.empty:
        continue
    prompting = subset['prompting'].iloc[0]
    ax.plot(
        subset['T'],
        subset['accuracy'] * 100,
        marker='o',
        linewidth=2,
        label=label,
        color=COLORS[prompting],
        alpha=0.55 if prompting == 'No CoT' else 1.0,
        linestyle='--' if prompting == 'No CoT' else '-',
    )
ax.set_xticks([2, 4, 6, 8, 12, 16])
ax.set_ylim(0, 105)
ax.set_xlabel('Target-relevant update depth, $T$')
ax.set_ylabel('Final-answer accuracy (%)')
ax.set_title('RQ1: Accuracy by temporal depth', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2, fontsize=8)
fig.tight_layout()
save_figure(fig, 'fig02_rq1_temporal_depth')

In [ ]:
rq3 = predictions.query("experiment == 'rq3_distractor' and family == 'interleaved_chain'").copy()
rq3_summary = accuracy_by(rq3, ['config_label', 'D'])
rq3_table = rq3_summary.pivot(index='config_label', columns='D', values='accuracy').reindex(CONFIG_LABELS)
fig, ax = plt.subplots(figsize=(8.6, 4.8))
image = ax.imshow(rq3_table * 100, cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(rq3_table.columns)), [f'D={int(value)}' for value in rq3_table.columns])
ax.set_yticks(range(len(rq3_table.index)), rq3_table.index)
for row_index in range(rq3_table.shape[0]):
    for column_index in range(rq3_table.shape[1]):
        value = rq3_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('State-changing distractor count, $D$')
ax.set_title('RQ3: Distractor accuracy heatmap', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig04_rq3_distractor_heatmap')

In [ ]:
metric_rows = []
for label in CONFIG_LABELS:
    subset = predictions.query('config_label == @label')
    final_accuracy = subset['is_correct'].mean()
    step_accuracy = subset['step_accuracy'].dropna().mean()
    if np.isfinite(step_accuracy):
        metric_rows.append({'config_label': label, 'metric': 'Final answer', 'accuracy': final_accuracy})
        metric_rows.append({'config_label': label, 'metric': 'Step-wise', 'accuracy': step_accuracy})
metric_table = pd.DataFrame(metric_rows)
fig, ax = plt.subplots(figsize=(9.2, 5.2))
labels = metric_table['config_label'].drop_duplicates().tolist()
x = np.arange(len(labels))
width = 0.36
for offset, metric, color in [(-width / 2, 'Final answer', '#0F766E'), (width / 2, 'Step-wise', '#E07A5F')]:
    values = [metric_table.query('config_label == @label and metric == @metric')['accuracy'].iloc[0] for label in labels]
    bars = ax.bar(x + offset, np.array(values) * 100, width, label=metric, color=color)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x, labels, rotation=25, ha='right')
ax.set_ylim(0, 58)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Final-answer versus canonical step-wise accuracy', loc='left', weight='bold')
ax.legend(frameon=False, ncols=2)
fig.tight_layout()
save_figure(fig, 'fig05_final_vs_stepwise')

In [ ]:
rq5 = predictions.query("experiment == 'rq5_pilot'").copy()
operation_order = ['split_chain', 'merge_chain', 'swap_chain', 'undo_chain', 'undo_redo_chain']
operation_labels = ['Split', 'Merge', 'Swap', 'Undo', 'Undo-Redo']
rq5_summary = accuracy_by(rq5, ['config_label', 'family'])
rq5_table = rq5_summary.pivot(index='config_label', columns='family', values='accuracy').reindex(index=CONFIG_LABELS, columns=operation_order)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
image = ax.imshow(rq5_table * 100, cmap='PuBuGn', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(operation_order)), operation_labels)
ax.set_yticks(range(len(rq5_table.index)), rq5_table.index)
for row_index in range(rq5_table.shape[0]):
    for column_index in range(rq5_table.shape[1]):
        value = rq5_table.iloc[row_index, column_index]
        if pd.notna(value):
            ax.text(column_index, row_index, f'{value * 100:.0f}', ha='center', va='center', fontsize=8)
ax.set_xlabel('Structural-operation family')
ax.set_title('RQ5: Operation-family accuracy heatmap', loc='left', weight='bold')
fig.colorbar(image, ax=ax, label='Final-answer accuracy (%)')
fig.tight_layout()
save_figure(fig, 'fig06_rq5_operation_heatmap')